# WiTA v2 — X-CLIP Video-Language Cross-Modal CTC (Kaggle T4)

Pivot from per-frame SigLIP (which mode-collapsed because each frame encodes 'person sitting', not the writing motion). X-CLIP encodes 8-frame video segments with multi-frame attention, so each feature captures motion within that window — exactly the signal WiTA needs.

| | |
|---|---|
| GPU | NVIDIA T4 (16 GB VRAM), ×2 |
| Encoder | X-CLIP base patch16 zero-shot (frozen, ~150M params) |
| Window | 8 frames per X-CLIP segment, stride=4 (~15 segs / 64-frame clip) |
| Prototypes | X-CLIP text encoder over per-char prompts (frozen) |
| Trainable | BiLSTM/Conv/Transformer adapter + Linear projection + log_inv_tau (~5M) |
| Head fix (vs SigLIP run) | L2-normed cosine sim, init_tau=1.0 (no logit saturation) |

**Pipeline:** install → setup → stream data → extract X-CLIP segment features ONCE → train adapter on cache → evaluate

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'transformers>=4.40' 'mediapipe>=0.10.0' --quiet

import sys, os

# Force re-clone so we pick up the latest commit on this branch.
!rm -rf /kaggle/working/wita_v2
!git clone -b video-language-cross-modal "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'

sys.path.insert(0, '/kaggle/working')


## Cell 2 — Imports & Config

In [ ]:
import os, sys, logging, random
import numpy as np
import torch

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/logs',        exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s \u2014 %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('/kaggle/working/logs/run.log'),
    ]
)

from wita_v2.configs.default import (
    Config, DataConfig, AugConfig, EncoderConfig,
    RecurrentConfig, AttnDecoderConfig, TrainConfig
)

# New cache path for the hand-crop variant.  Old cache (without crop) lives
# at /kaggle/working/xclip_features.pt and can be kept for ablation.
FEATURE_CACHE = '/kaggle/working/xclip_features_handcrop.pt'

cfg = Config(
    data=DataConfig(
        hf_repo_id  = 'yewon816/WiTA',
        lang        = 'english',
        max_zips    = None,        # \u2190 set small (e.g. 4) for a smoke test
        max_frames  = 64,
    ),
    aug=AugConfig(),               # unused \u2014 cached features cannot be augmented
    encoder=EncoderConfig(
        arch                  = 'xclip',
        xclip_model_name      = 'microsoft/xclip-base-patch16-zero-shot',
        img_size              = 224,
        out_dim               = 512,
        xclip_num_frames      = 8,
        xclip_stride          = 4,
        xclip_char_template   = 'writing the letter {ch} in the air',
        xclip_blank_template  = 'no writing',
        xclip_sep_template    = 'a brief pause between letters',
        # \u2500\u2500 Path A: MediaPipe hand crop \u2500\u2500
        enable_hand_crop      = True,
        hand_crop_padding     = 0.3,
        hand_crop_min_conf    = 0.3,
        # \u2500\u2500 Head fields (shared with SigLIP path) \u2500\u2500
        siglip_temporal_arch  = 'lstm',
        siglip_adapter_hidden = 512,
        siglip_adapter_layers = 1,
        siglip_dropout        = 0.1,
        siglip_init_tau       = 1.0,
        siglip_learnable_tau  = True,
    ),
    recurrent=RecurrentConfig(),
    attn=AttnDecoderConfig(),
    train=TrainConfig(
        num_epochs           = 80,
        batch_size           = 32,
        accum_steps          = 1,
        lr                   = 1e-3,
        beta2                = 0.999,
        num_workers          = 2,
        optimizer            = 'adamw',
        scheduler            = 'onecycle',
        val_limit            = None,
        grad_clip            = 5.0,
        unfreeze_after_epoch = 999,
        backbone_lr_mult     = 1.0,
        checkpoint_dir       = '/kaggle/working/checkpoints',
        resume_path          = None,
        save_frequency       = 10,
        seed                 = 42,
        warmup_pct           = 0.05,
        xclip_feature_cache  = FEATURE_CACHE,
    ),
).build()

random.seed(cfg.train.seed)
np.random.seed(cfg.train.seed)
torch.manual_seed(cfg.train.seed)
torch.backends.cudnn.benchmark = True

print(f'Device          : {cfg.device}')
print(f'X-CLIP model    : {cfg.encoder.xclip_model_name}')
print(f'Window / stride : {cfg.encoder.xclip_num_frames} frames / stride {cfg.encoder.xclip_stride}')
print(f'Visual dim      : {cfg.encoder.out_dim}')
print(f'Hand crop       : {cfg.encoder.enable_hand_crop}  (padding={cfg.encoder.hand_crop_padding})')
print(f'Adapter         : {cfg.encoder.siglip_temporal_arch}  hidden={cfg.encoder.siglip_adapter_hidden}')
print(f'init_tau        : {cfg.encoder.siglip_init_tau}')
print(f'CTC vocab       : {cfg.vocab.ctc_vocab_size}  (blank=0, sep={cfg.vocab.sep_idx})')
print(f'Cache path      : {cfg.train.xclip_feature_cache}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)} \u00d7 {torch.cuda.device_count()}')


## Cell 3 — Stream + index data

In [ ]:
from wita_v2.datasets.dataset import stream_and_index
from wita_v2.datasets.vocab   import make_converter

converter = make_converter(cfg.data.lang)
samples   = stream_and_index(cfg)
print(f'Total samples indexed: {len(samples)}')

## Cell 4 — Extract X-CLIP video features (ONCE)

Each clip is broken into sliding 8-frame windows (stride 4) and each window is encoded through X-CLIP's video encoder. Output cache shape: per-clip [N_seg, 512]. Re-run after deleting the cache file to force re-extraction with different settings.

In [ ]:
from wita_v2.models.encoders.xclip_encoder    import XCLIPVideoEncoder
from wita_v2.datasets.video_feature_cache     import extract_xclip_features

if os.path.exists(FEATURE_CACHE):
    print(f'Cache already exists: {FEATURE_CACHE}  '
          f'({os.path.getsize(FEATURE_CACHE) / 1e6:.1f} MB) \u2014 skipping extraction.')
else:
    # Build the hand cropper (Path A).  Same instance is used at deployment.
    hand_cropper = None
    if cfg.encoder.enable_hand_crop:
        from wita_v2.datasets.hand_crop import HandCropper
        hand_cropper = HandCropper(
            target_size              = cfg.encoder.img_size,
            padding_ratio            = cfg.encoder.hand_crop_padding,
            min_detection_confidence = cfg.encoder.hand_crop_min_conf,
            min_tracking_confidence  = cfg.encoder.hand_crop_min_conf,
            max_num_hands            = 1,
        )

    encoder = XCLIPVideoEncoder(
        model_name = cfg.encoder.xclip_model_name,
        image_size = cfg.encoder.img_size,
        num_frames = cfg.encoder.xclip_num_frames,
        stride     = cfg.encoder.xclip_stride,
    )
    cache = extract_xclip_features(
        samples      = samples,
        encoder      = encoder,
        out_path     = FEATURE_CACHE,
        seg_chunk    = 4,
        device       = cfg.device,
        dtype        = torch.float16,
        hand_cropper = hand_cropper,
    )
    if hand_cropper is not None:
        s = hand_cropper.stats()
        print(f"Hand crop stats: clips_seen={s['clips_seen']}  "
              f"no_hand={s['clips_no_hand']}  partial={s['clips_partial']}  "
              f"frame_detect_rate={s['frame_detect_rate']*100:.1f}%")
        hand_cropper.close()
    del encoder, cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Cell 4b \u2014 Visualize hand crop (sanity check)

Pick a random sample, show the raw frames and the hand-cropped frames side by side.  If the crops don't contain the writing hand, MediaPipe is losing it \u2014 lower `hand_crop_min_conf` or adjust `hand_crop_padding`.

In [ ]:
import io, random
import matplotlib.pyplot as plt
from PIL import Image
from wita_v2.datasets.hand_crop import HandCropper

# Spin up a temporary cropper just for visualization (the one above was closed).
viz_cropper = HandCropper(
    target_size              = cfg.encoder.img_size,
    padding_ratio            = cfg.encoder.hand_crop_padding,
    min_detection_confidence = cfg.encoder.hand_crop_min_conf,
    min_tracking_confidence  = cfg.encoder.hand_crop_min_conf,
)

idx = random.randint(0, len(samples) - 1)
frame_bytes, label = samples[idx]
raw_frames = [Image.open(io.BytesIO(b)).convert('RGB') for b in frame_bytes]
cropped    = viz_cropper.crop_clip(raw_frames)

print(f"label='{label}'  T={len(raw_frames)}  raw_size={raw_frames[0].size}  cropped_size={cropped[0].size}")
print(f"detect_rate this clip: {viz_cropper.stats()['frame_detect_rate']*100:.1f}%")

# Show 6 evenly-spaced frames
import numpy as np
pick = np.linspace(0, len(raw_frames) - 1, 6).round().astype(int)
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for j, t in enumerate(pick):
    axes[0, j].imshow(raw_frames[t]);    axes[0, j].axis('off'); axes[0, j].set_title(f'raw[{t}]', fontsize=8)
    axes[1, j].imshow(cropped[t]);       axes[1, j].axis('off'); axes[1, j].set_title(f'cropped[{t}]', fontsize=8)
axes[0, 0].set_ylabel('RAW',     fontsize=12, rotation=0, labelpad=40)
axes[1, 0].set_ylabel('CROPPED', fontsize=12, rotation=0, labelpad=40)
fig.suptitle(f"label='{label}'", fontsize=14)
plt.tight_layout()
plt.show()
viz_cropper.close()


## Cell 5 — Build DataLoaders over the X-CLIP cache

Reuses the SAME `make_cached_dataloaders` from the SigLIP path — the cache format is identical (variable-length [T, D] tensors).

In [ ]:
from wita_v2.datasets.feature_cache import make_cached_dataloaders

train_loader, val_loader, cache_meta = make_cached_dataloaders(
    cfg, FEATURE_CACHE, converter=converter,
)
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Source model  : {cache_meta["model_name"]}')
print(f'Feature dim   : {cache_meta["out_dim"]}')
print(f'X-CLIP window : {cache_meta.get("num_frames")} frames, stride {cache_meta.get("stride")}')

feats, labels, input_lens, label_lens = next(iter(train_loader))
print(f'feats         : {feats.shape}  {feats.dtype}    (B, T_seg, D)')
print(f'labels        : {labels.shape} {labels.dtype}')
print(f'input_lens    : {input_lens.tolist()[:8]}  (segments per clip)')
print(f'label_lens    : {label_lens.tolist()[:8]}')

gt = converter.decode(
    labels[0, :label_lens[0].item()].int(),
    torch.IntTensor([label_lens[0].item()]),
)
print(f'GT word[0]    : "{gt}"')

## Cell 6 — Build cross-modal CTC model

Uses the same `WiTACLIPCTCModel` as the SigLIP path. It auto-dispatches prototype building to X-CLIP's text encoder based on `cfg.encoder.arch`.

In [ ]:
from wita_v2.models.clip_ctc_model import WiTACLIPCTCModel

cfg.encoder.out_dim = cache_meta['out_dim']  # sync defensively

model = WiTACLIPCTCModel(cfg).to(cfg.device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}  (X-CLIP frozen, prototypes frozen)')

feats_b      = feats.to(cfg.device)
input_lens_b = input_lens.to(cfg.device)
with torch.no_grad():
    ctc_lp, enc_lens = model(feats_b, input_lens_b)
print(f'CTC log_probs   : {ctc_lp.shape}  (B, T_seg, V)')
print(f'enc_lens        : {enc_lens.tolist()[:8]}')

# Sanity check: initial CTC loss should be near log(V) * avg_label_len, not 100+.
import torch.nn as nn
with torch.no_grad():
    sane_loss = nn.functional.ctc_loss(
        ctc_lp.transpose(0, 1).float(),
        labels.to(cfg.device),
        enc_lens, label_lens.to(cfg.device),
        blank=0, zero_infinity=True,
    )
print(f'Initial CTC loss: {sane_loss.item():.4f}   (target: ≈ log({cfg.vocab.ctc_vocab_size}) × avg_label_len ≈ 3.3 × 7 ≈ 23)')
if torch.cuda.is_available():
    print(f'VRAM after fwd  : {torch.cuda.memory_allocated()/1e6:.0f} MB')

## Cell 7 — Train

In [ ]:
from wita_v2.training.trainer import train

best_model = train(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    converter    = converter,
    cfg          = cfg,
)
print('Training complete.')

## Cell 8 — Final evaluation

In [ ]:
from wita_v2.evaluation.evaluator import evaluate_cer, print_sample_table

print('Running full validation (no batch limit) …')
cer, _ = evaluate_cer(
    best_model, val_loader, converter, cfg,
    decode_mode='ctc', max_batches=None,
)
print(f'Final Val CER : {cer:.4f}')

print_sample_table(
    best_model, val_loader, converter, cfg,
    epoch=cfg.train.num_epochs, max_batches=None,
)

print()
print('─' * 50)
print('Reference baselines')
print('  Original WiTA paper (R3D-18 + CTC)        : 0.2924')
print('  SigLIP per-frame (Run-7 best)             : 0.8778')
print(f'  This run (X-CLIP video-language)          : {cer:.4f}')

## Cell 9 — Ablations

Edit and re-run Cells 6 → 7 → 8. The cached features stay valid for most ablations.

In [ ]:
# ── Adapter ─────────────────────────────────────────────────────────
# cfg.encoder.siglip_temporal_arch = 'transformer'
# cfg.encoder.siglip_temporal_arch = 'conv'
# cfg.encoder.siglip_temporal_arch = 'none'

# ── Prompt template (need to re-init model — prototypes change) ─────
# cfg.encoder.xclip_char_template = 'a person writing the letter {ch}'
# cfg.encoder.xclip_char_template = 'the letter {ch}'

# ── Temperature ─────────────────────────────────────────────────────
# cfg.encoder.siglip_init_tau     = 0.1
# cfg.encoder.siglip_learnable_tau = False

# ── X-CLIP window stride (requires cache re-extraction) ─────────────
# cfg.encoder.xclip_stride = 2     # denser segments, more CTC timesteps
# cfg.encoder.xclip_stride = 8     # no overlap, fewer segments

# ── Adapter capacity ────────────────────────────────────────────────
# cfg.encoder.siglip_adapter_layers = 2
# cfg.encoder.siglip_adapter_hidden = 1024

print('Edit this cell and re-run Cells 6 → 7 → 8 to run an ablation.')